In [28]:
import os
import json
import pickle
import joblib
import csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Folder to store all the scalers and models
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [29]:
# LOAD DATA
print("Loading JSON files...")
with open('fulldb-entities.json', 'r') as db_json:
    db = json.load(db_json)
with open('processed_labels/removal_labels.json', 'r') as labels_json:
    labels = json.load(labels_json)
with open('processed_labels/test_sample.json', 'r') as test_file:
    test = json.load(test_file)

dataset_cuis = sorted([cui for cui in labels.keys() if cui in db and cui not in test])
print(f"Unique Entities in train set = {len(dataset_cuis)}")

test_cuis = sorted([cui for cui in test.keys() if cui in db])
print(f"Unique Entities in test set  = {len(test_cuis)}")

Loading JSON files...
Unique Entities in train set = 1055
Unique Entities in test set  = 76


In [30]:
# PATCH SEMANTIC TYPES FROM CSV
print("\nPatching Semantic Types from combined.csv...")
# We read the CSV to grab the semantic types since they were missing in Neo4j
try:
    df_combined = pd.read_csv("annotated/combined.csv", sep=None, engine='python', encoding='utf-8-sig')
    
    # Quick sanity check to see what columns Pandas found
    if 'CUI' not in df_combined.columns:
        print(f"ERROR: 'CUI' column not found. Found these instead: {df_combined.columns.tolist()}")
    else:
        for index, row in df_combined.iterrows():
            cui = str(row['CUI']).strip()
            
            if cui in db:
                raw_sty = str(row['Semantic Types'])
                
                # Clean the string (e.g., turn '["neop"]' into 'neop')
                clean_sty = raw_sty.replace('[', '').replace(']', '').replace('"', '').replace("'", "")
                sty_list = [s.strip() for s in clean_sty.split(',') if s.strip()]
                
                db[cui]["sem_types"] = sty_list
                
        print("Patch complete!")

except Exception as e:
    print(f"Could not read combined.csv: {e}")


Patching Semantic Types from combined.csv...
Patch complete!


In [31]:
# SEMANTIC TYPES & L1 FEATURE SELECTION
print("\nExtracting and selecting Semantic Types...")
sem_types = []
for cui in db.keys():
    for sem_type in db[cui]["sem_types"]:
        if sem_type not in sem_types:
            sem_types.append(sem_type)

sem_types_one_hot = []
for cui in dataset_cuis:
    row = [1 if st in db[cui]["sem_types"] else 0 for st in sem_types]
    sem_types_one_hot.append(row)
X_sem = np.asarray(sem_types_one_hot)

Y = np.asarray([labels[cui] for cui in dataset_cuis])

# Use L1 to find important semantic types
coefs = LogisticRegression(penalty='l1', solver='liblinear', random_state=2020).fit(X_sem, Y).coef_[-1]
sem_type_imporances = sorted(list(zip(sem_types, coefs)), key=lambda x: abs(x[1]), reverse=True)

selected_sem_types = []
for sem_type, coef in sem_type_imporances:
    if abs(coef) > 0:
        selected_sem_types.append(sem_type)

print(f"Reduced from {len(sem_types)} to {len(selected_sem_types)} important Semantic Types.")
pickle.dump(selected_sem_types, open(f"{ARTIFACT_DIR}/selected_sem_types.pkl", "wb"))
joblib.dump(selected_sem_types, f"{ARTIFACT_DIR}/selected_sem_types.joblib")

selected_sem_types_one_hot = []
for cui in dataset_cuis:
    row = [1 if st in db[cui]["sem_types"] else 0 for st in selected_sem_types]
    selected_sem_types_one_hot.append(row)
X_selected_sem = np.asarray(selected_sem_types_one_hot)


Extracting and selecting Semantic Types...
Reduced from 96 to 53 important Semantic Types.


In [32]:
# PREPARE CENTRALITY & EMBEDDINGS
print("\nScaling Centrality & Graph Embeddings...")

# PageRank
pagerank = {cui: db[cui].get("pagerank", 0) for cui in dataset_cuis}
X_pagerank = np.asarray([[pagerank[cui] if pagerank[cui] else 0] for cui in dataset_cuis])
pagerank_scaler = MinMaxScaler()
X_pagerank = pagerank_scaler.fit_transform(X_pagerank)
joblib.dump(pagerank_scaler, f"{ARTIFACT_DIR}/pagerank_scaler.joblib")

# 2. HITS Authority & Hub
authscore = {cui: db[cui].get("authscore", 0) for cui in dataset_cuis}
X_authscore = np.asarray([[authscore[cui] if authscore[cui] else 0] for cui in dataset_cuis])
auth_scaler = MinMaxScaler()
X_authscore = auth_scaler.fit_transform(X_authscore)
joblib.dump(auth_scaler, f"{ARTIFACT_DIR}/auth_scaler.joblib")

hubscore = {cui: db[cui].get("hubscore", 0) for cui in dataset_cuis}
X_hubscore = np.asarray([[hubscore[cui] if hubscore[cui] else 0] for cui in dataset_cuis])
hub_scaler = MinMaxScaler()
X_hubscore = hub_scaler.fit_transform(X_hubscore)
joblib.dump(hub_scaler, f"{ARTIFACT_DIR}/hub_scaler.joblib")

# Combine HITS for easy testing
X_hits = np.concatenate([X_authscore, X_hubscore], axis=1)

# ==========================================
# PREPARE LOUVAIN COMMUNITIES (One-Hot)
# ==========================================
print("Extracting and encoding Louvain Communities...")

# Find all unique communities present in the training set
train_communities = set()
for cui in dataset_cuis:
    comm_id = db[cui].get("communityId")
    if comm_id is not None:
        train_communities.add(comm_id)

# Convert to a sorted list so the column order is always exactly the same
train_communities = sorted(list(train_communities))
print(f"Found {len(train_communities)} distinct Louvain communities in the training data.")

# Save the list of communities so the Test set knows exactly how many columns to make
joblib.dump(train_communities, f"{ARTIFACT_DIR}/louvain_communities.joblib")

# Build the One-Hot Matrix for Training
louvain_one_hot = []
for cui in dataset_cuis:
    node_comm = db[cui].get("communityId")
    # Create a row of 0s, and put a 1 in the exact slot for this node's community
    row = [1 if node_comm == comm else 0 for comm in train_communities]
    louvain_one_hot.append(row)

X_louvain = np.asarray(louvain_one_hot)

# Embeddings (The Tournament)
algos = ["node2vec", "fastrp", "graphsage", "hashgnn"]
dims = ["32", "64", "128"]

X_embeddings = {}
embedding_scalers = {}

for algo in algos:
    for dim in dims:
        key = f"{algo}{dim}"
        # Fetch the list of numbers from JSON. If missing, create a zero array.
        X_embeddings[key] = np.asarray([db[cui].get(key, [0]*int(dim)) for cui in dataset_cuis])
        
        # Scale
        scaler = MinMaxScaler()
        X_embeddings[key] = scaler.fit_transform(X_embeddings[key])
        joblib.dump(scaler, f"{ARTIFACT_DIR}/{key}_scaler.joblib")


Scaling Centrality & Graph Embeddings...
Extracting and encoding Louvain Communities...
Found 685 distinct Louvain communities in the training data.


In [33]:
# BUILD COMBINATIONS (Gyftakis + New Algos)

feature_combinations = {}

# --- Baselines (From Table 4.1) ---
feature_combinations["sem_types"] = X_sem
feature_combinations["sem_types + pagerank"] = np.concatenate([X_sem, X_pagerank], axis=1)
feature_combinations["selected_sem_types"] = X_selected_sem
feature_combinations["selected_sem_types + pagerank"] = np.concatenate([X_selected_sem, X_pagerank], axis=1)

# --- New Centrality Baselines ---
feature_combinations["hits"] = X_hits
feature_combinations["selected_sem_types + hits"] = np.concatenate([X_selected_sem, X_hits], axis=1)

# --- Baseline Louvain Tests ---
feature_combinations["louvain"] = X_louvain
feature_combinations["louvain + selected_sem_types"] = np.concatenate([X_louvain, X_selected_sem], axis=1)

# --- Dynamic Embedding Grid ---
for algo in algos:
    for dim in dims:
        base = f"{algo}{dim}"
        emb = X_embeddings[base]
        
        # 1. Base Embedding (Table 4.1)
        feature_combinations[f"{base}"] = emb
        
        # 2. Emb + PageRank (Table 4.1)
        feature_combinations[f"{base} + pagerank"] = np.concatenate([emb, X_pagerank], axis=1)
        
        # 3. Emb + All Sem (Table 4.1)
        feature_combinations[f"{base} + sem_types"] = np.concatenate([emb, X_sem], axis=1)
        
        # 4. Emb + All Sem + PageRank (Table 4.1)
        feature_combinations[f"{base} + sem_types + pagerank"] = np.concatenate([emb, X_sem, X_pagerank], axis=1)
        
        # 5. Emb + Selected Sem (Table 4.1)
        feature_combinations[f"{base} + selected_sem_types"] = np.concatenate([emb, X_selected_sem], axis=1)
        
        # 6. Emb + Selected Sem + PageRank (Table 4.1)
        feature_combinations[f"{base} + selected_sem_types + pagerank"] = np.concatenate([emb, X_selected_sem, X_pagerank], axis=1)
        
        # 7. NEW: Emb + HITS
        feature_combinations[f"{base} + hits"] = np.concatenate([emb, X_hits], axis=1)
        
        # 8. NEW: Emb + Selected Sem + HITS
        feature_combinations[f"{base} + selected_sem_types + hits"] = np.concatenate([emb, X_selected_sem, X_hits], axis=1)
        
        # 9. NEW: The Ultimate Combo (All Centrality)
        feature_combinations[f"{base} + selected_sem_types + pagerank + hits"] = np.concatenate([emb, X_selected_sem, X_pagerank, X_hits], axis=1)

        # 10. NEW: Emb + Louvain
        feature_combinations[f"{base} + louvain"] = np.concatenate([emb, X_louvain], axis=1)

        # 11. NEW: Emb + Selected Sem + Louvain
        feature_combinations[f"{base} + selected_sem_types + louvain"] = np.concatenate([emb, X_selected_sem, X_louvain], axis=1)

In [34]:
# RUN CROSS-VALIDATION TOURNAMENT
print("\nRunning 10-Fold CV Tournament... ")
feature_scores = {}
scoring = 'f1_micro'

print(f"{'Features':<35} | {'Dims':<4} | {'F1 Score':<15}")
print("-" * 60)

for name, X in feature_combinations.items():
    cv_results = cross_validate(
        LogisticRegression(penalty='l2', solver='liblinear', random_state=2020),
        X, Y, cv=10, scoring=scoring
    )
    median_f1 = np.median(cv_results['test_score'])
    feature_scores[name] = median_f1
    print(f"{name:<35} | {X.shape[1]:<4} | {median_f1:.3f} (+/-{np.std(cv_results['test_score']):.3f})")

# Find the absolute best
best_combo_name = max(feature_scores, key=feature_scores.get)
print("-" * 60)
print(f"WINNER: {best_combo_name} with F1 = {feature_scores[best_combo_name]:.3f}")


Running 10-Fold CV Tournament... 
Features                            | Dims | F1 Score       
------------------------------------------------------------
sem_types                           | 96   | 0.654 (+/-0.094)


sem_types + pagerank                | 97   | 0.654 (+/-0.094)
selected_sem_types                  | 53   | 0.654 (+/-0.095)
selected_sem_types + pagerank       | 54   | 0.654 (+/-0.095)
hits                                | 2    | 0.519 (+/-0.016)
selected_sem_types + hits           | 55   | 0.654 (+/-0.094)
louvain                             | 685  | 0.652 (+/-0.075)
louvain + selected_sem_types        | 738  | 0.676 (+/-0.086)
node2vec32                          | 32   | 0.630 (+/-0.073)
node2vec32 + pagerank               | 33   | 0.635 (+/-0.072)
node2vec32 + sem_types              | 128  | 0.668 (+/-0.089)
node2vec32 + sem_types + pagerank   | 129  | 0.663 (+/-0.089)
node2vec32 + selected_sem_types     | 85   | 0.668 (+/-0.093)
node2vec32 + selected_sem_types + pagerank | 86   | 0.668 (+/-0.093)
node2vec32 + hits                   | 34   | 0.635 (+/-0.073)
node2vec32 + selected_sem_types + hits | 87   | 0.668 (+/-0.094)
node2vec32 + selected_sem_types + pagerank + hits | 88   | 0

warning! den exw asxolhthei me tis parametrous gia ta parapanw runs + vgainoun polla nodes xwris relationships, [Pagerank 0] (isws kwn/nou)

In [35]:
# ==========================================
# EXPORT DYNAMIC CSV FILES
# ==========================================
print(f"\nExporting train_df.csv and test_df.csv using the winning combination: {best_combo_name}")
X_best = feature_combinations[best_combo_name]

# --- Smart Column Namer ---
algo_base = best_combo_name.split(" + ")[0]
dim = int(''.join(filter(str.isdigit, algo_base))) if any(char.isdigit() for char in algo_base) else 0

actual_feature_names = []

# 1. Embedding column names
if algo_base in X_embeddings:
    actual_feature_names += [f"{algo_base}_{i}" for i in range(1, dim + 1)]

# 2. Semantic Type column names
if "sem_types" in best_combo_name:
    actual_feature_names += sem_types if "selected" not in best_combo_name else selected_sem_types

# 3. PageRank column name
if "pagerank" in best_combo_name or "all_centrality" in best_combo_name:
    actual_feature_names += ["pagerank"]

# 4. HITS column names
if "hits" in best_combo_name or "all_centrality" in best_combo_name:
    actual_feature_names += ["authscore", "hubscore"]

# 5. Louvain column names (MUST happen before building train_df)
if "louvain" in best_combo_name:
    saved_communities = joblib.load(f"{ARTIFACT_DIR}/louvain_communities.joblib")
    actual_feature_names += [f"community_{comm}" for comm in saved_communities]

column_names = ['label'] + actual_feature_names + ['target']

# Train DataFrame
data_train = []
for ii, cui in enumerate(dataset_cuis):
    data_train.append([db[cui]['label']] + X_best[ii, :].tolist() + [Y[ii]])
train_df = pd.DataFrame(data=data_train, columns=column_names, index=dataset_cuis)
train_df.to_csv("./train_df.csv")

# ==========================================
# PREPARE TEST SET BASED ON WINNER
# ==========================================
X_test_parts = []

# 1. EMBEDDINGS FIRST
if algo_base in X_embeddings:
    test_emb = np.asarray([db[cui].get(algo_base, [0]*dim) for cui in test_cuis])
    emb_scaler = joblib.load(f"{ARTIFACT_DIR}/{algo_base}_scaler.joblib")
    X_test_parts.append(emb_scaler.transform(test_emb))

# 2. SEMANTIC TYPES SECOND
if "sem_types" in best_combo_name:
    ref_list = sem_types if "selected" not in best_combo_name else selected_sem_types
    test_sem_hot = [[1 if st in db[cui].get("sem_types", []) else 0 for st in ref_list] for cui in test_cuis]
    X_test_parts.append(np.asarray(test_sem_hot))

# 3. PAGERANK THIRD
if "pagerank" in best_combo_name or "all_centrality" in best_combo_name:
    test_pagerank = np.asarray([[db[cui].get("pagerank", 0) if db[cui].get("pagerank") else 0] for cui in test_cuis])
    X_test_parts.append(pagerank_scaler.transform(test_pagerank))

# 4. HITS FOURTH
if "hits" in best_combo_name or "all_centrality" in best_combo_name:
    test_auth = np.asarray([[db[cui].get("authscore", 0) if db[cui].get("authscore") else 0] for cui in test_cuis])
    test_hub = np.asarray([[db[cui].get("hubscore", 0) if db[cui].get("hubscore") else 0] for cui in test_cuis])
    X_test_parts.append(auth_scaler.transform(test_auth))
    X_test_parts.append(hub_scaler.transform(test_hub))

# 5. LOUVAIN COMMUNITIES FIFTH
if "louvain" in best_combo_name:
    test_louvain_hot = []
    for cui in test_cuis:
        node_comm = db[cui].get("communityId")
        row = [1 if node_comm == comm else 0 for comm in saved_communities]
        test_louvain_hot.append(row)
    X_test_parts.append(np.asarray(test_louvain_hot))

# Safely Concatenate
X_test_final = np.concatenate(X_test_parts, axis=1) if len(X_test_parts) > 1 else X_test_parts[0]

data_test = []
for ii, cui in enumerate(test_cuis):
    data_test.append([db[cui]['label']] + X_test_final[ii, :].tolist() + [test[cui]])
    
test_df = pd.DataFrame(data=data_test, columns=column_names, index=test_cuis)
test_df.to_csv("./test_df.csv")

print(f"Success! Exported {train_df.shape[1]} columns to train_df.csv and test_df.csv.")


Exporting train_df.csv and test_df.csv using the winning combination: hashgnn32 + selected_sem_types + louvain
Success! Exported 772 columns to train_df.csv and test_df.csv.


In [36]:
# ==========================================
# EXPORT TOURNAMENT RESULTS TO EXCEL (.xlsx)
# ==========================================
print("Exporting tournament results to Excel...")

# 1. Convert the dictionary into a list of dictionaries for Pandas
results_list = []
for name, score in feature_scores.items():
    # Attempt to calculate the exact number of dimensions for the Excel report
    try:
        dims = feature_combinations[name].shape[1]
    except:
        dims = "Unknown"
        
    results_list.append({
        "Feature Combination": name,
        "Dimensions": dims,
        "F1 Score": round(score, 4)
    })

# 2. Create the DataFrame and sort it by the highest score
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

# 3. Export to a true Excel file (Requires 'openpyxl' installed)
excel_path = f"{ARTIFACT_DIR}/tournament_results.xlsx"
results_df.to_excel(excel_path, index=False, sheet_name="Tournament Results")

print(f"Success! Open '{excel_path}' in Excel.")

Exporting tournament results to Excel...
Success! Open 'artifacts/tournament_results.xlsx' in Excel.
